In [1]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os, requests
from dotenv import load_dotenv
load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

1526***


In [32]:
# Q1
# (a) - (1)
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    params = {
        "key": KEY, "q": q, "req_type": "json",
        "num": num, "start": start, "type1": "word",
    }
    r = requests.get("https://stdict.korean.go.kr/api/search.do", params=params, timeout = 10)
    r.raise_for_status()
    return r.json()

(a) - (2)
출력 없음. 함수 search_word가 정의되어 메모리에 등록되었다.
(a) - (3)
search_word는 우리말샘 API 엔드포인트에 requests.get으로 GET 요청을 보내 JSON 응답을 반환하는 함수다. 쿼리 파라미터로 key, q, req_type, num, start, type1을 전달하며, timeout=10으로 무한 대기를 방지한다. raise_for_status()를 통해 HTTP 오류 발생 시 즉시 예외를 일으켜 잘못된 응답이 그대로 반환되는 것을 막는다.

In [ ]:
# (b) - (1)
import json

data = search_word("김치")
print(json.dumps(data,ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 3,
    "num": 10,
    "title": "표준 국어 대사전 개발 지원(Open API) - 사전  검색",
    "start": 1,
    "description": "표준 국어 대사전 개발 지원(Open API) – 사전 검색 결과",
    "item": [
      {
        "sup_no": "1",
        "origin": "",
        "word": "김치",
        "target_code": "52827",
        "sense": {
          "definition": "소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린 뒤 발효를 시킨 음식. 재료와 조리 


(b) - (2)
{
  "channel": {
    "total": 3,
    "num": 10,
    "title": "표준 국어 대사전 개발 지원(Open API) - 사전  검색",
    "start": 1,
    "description": "표준 국어 대사전 개발 지원(Open API) – 사전 검색 결과",
    "item": [
      {
        "sup_no": "1",
        "origin": "",
        "word": "김치",
        "target_code": "52827",
        "sense": {
          "definition": "소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린 뒤 발효를 시킨 음식. 재료와 조리
(b) - (3)
json.dumps(data, ensure_ascii=False, indent=2)는 응답 JSON을 들여쓰기 2칸으로 사람이 읽기 좋게 출력한다. ensure_ascii=False를 빼면 한글이 \uae40\uce58와 같은 유니코드 이스케이프 시퀀스로 출력되어 내용을 확인하기 어렵다. 앞 400자만 슬라이싱한 이유는 응답 전체가 길어 출력이 과도하게 길어지는 것을 방지하기 위해서다.

In [ ]:
# (c) - (1)
# (i)
items = data["channel"]["item"]
total = data["channel"]["total"]
n = len(items)

print(f"총 {total}건, 이 페이지 {n}건")
# (ii)
for item in items[:5]:
    word: str = item["word"]
    pos: str = item.get("pos","품사 없음")
    sense: str = item["sense"]["definition"]
    print(f"{word} ({pos}): -> {sense[:40]}")

총 3건, 이 페이지 3건
김치 (명사): -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사): -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사): -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南


(c) - (2)
총 3건, 이 페이지 3건
김치 (명사): -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사): -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사): -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
(c) - (3)
data["channel"]["total"]에서 전체 검색 결과 수를, len(items)로 현재 페이지에서 수신한 항목 수를 각각 추출하여 출력했다. pos 필드는 모든 항목에 존재하지 않으므로 item.get("pos", "품사 없음")으로 안전하게 접근했다. 뜻풀이는 item["sense"]["definition"]로 추출한 뒤 앞 40자만 슬라이싱하여 출력이 한 줄에 들어오도록 했다.

In [ ]:
# Q2
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]
# (a) - (1)
all_items = []
for word in words:
    data = search_word(word)
    total = data["channel"]["total"]
    items = data["channel"]["item"]
    all_items.extend(items)
    print(f"{word}: {total}건")
    time.sleep(0.3)

# (b) - (1)
pos_counter = Counter(item.get("pos") or "(미상)" for item in all_items)

print("\n품사 빈도 상위 3개:")
for pos, count in pos_counter.most_common(3):
    print(f"{pos}: {count}건")

    

김치: 3건
라면: 2건
만두: 2건
김밥: 1건
국수: 6건
떡볶이: 1건
불고기: 1건
비빔밥: 1건

품사 빈도 상위 3개:
명사: 16건
어미: 1건


(a) - (2)
김치: 3건
라면: 2건
만두: 2건
김밥: 1건
국수: 6건
떡볶이: 1건
불고기: 1건
비빔밥: 1건
(a) - (3)
8개 음식 관련 검색어에 대해 search_word를 순차적으로 호출하여 각 검색어의 전체 결과 수를 출력했다. 매 호출 사이에 time.sleep(0.3)을 삽입하여 짧은 시간에 요청이 집중되지 않도록 서버 부하를 줄였다. 수집한 항목은 all_items에 누적하여 (b)에서 재사용했다.
(b) - (2)
품사 빈도 상위 3개:
명사: 16건
어미: 1건
(b) - (3)
Counter로 all_items 전체의 pos 필드를 집계하고 most_common(3)으로 빈도 상위 3개를 출력했다. item.get("pos") or "(미상)"을 사용하여 필드가 없거나 빈 문자열인 경우를 모두 "(미상)"으로 통일했다. 가장 흔한 품사는 명사(16건)로, 음식 관련 어휘는 대부분 사물을 지칭하는 명사로 사전에 등재되기 때문이다. 어미가 1건 등장한 것은 "라면"처럼 어미 "-라면"과 표제어가 겹치는 경우가 포함된 결과로 보인다.